# 06 — Measuring Performance Correctly

**Master syllabus coverage:** V2 1.6

## Why this matters

Performance claims guide architecture and runtime decisions only when measurements are synchronized, repeated, reproducible, equivalent, and connected to a product-relevant metric.

## Learning objectives

- Build a warm-up, synchronization, repetition, and percentile benchmark harness.
- Separate cold-start from steady-state latency.
- Convert time into qualified operation or bandwidth throughput.
- Produce a reproducible benchmark report with correctness evidence.

Work through the notebook in order. Predict shapes and results before running each code cell. An assertion failure is part of the lesson: read it before changing code.


## 1. A benchmark is an experiment with a declared question

State the operation, shapes, dtype, device, setup boundary, warm-up, synchronization,
repeats, statistic, and correctness tolerance. Latency, throughput, peak memory, and
energy answer different questions. Profiling explains where time goes; benchmarking
compares an outcome under controlled conditions.


In [ ]:
import platform
import time
import numpy as np
import torch

def synchronize(device: torch.device) -> None:
    if device.type == "cuda": torch.cuda.synchronize(device)
    elif device.type == "mps": torch.mps.synchronize()

def benchmark(function, *, warmup=5, repeats=30, device=torch.device("cpu")):
    for _ in range(warmup): function()
    synchronize(device)
    samples = []
    for _ in range(repeats):
        synchronize(device)
        start = time.perf_counter_ns()
        function()
        synchronize(device)
        samples.append((time.perf_counter_ns() - start) / 1e6)
    values = np.asarray(samples)
    return {
        "median_ms": float(np.median(values)),
        "p90_ms": float(np.percentile(values, 90)),
        "min_ms": float(values.min()),
        "max_ms": float(values.max()),
        "repeats": repeats,
    }


## 2. Separate setup from steady-state work

First use can include imports, allocation, graph/kernel compilation, cache filling, and
autotuning. Excluding setup is correct for steady-state questions but wrong for cold-start
latency. Report both when the product experiences both.


In [ ]:
a = torch.randn(512, 512)
b = torch.randn(512, 512)
cold_start = benchmark(lambda: a @ b, warmup=0, repeats=1)
steady_state = benchmark(lambda: a @ b, warmup=5, repeats=20)
print("cold:", cold_start)
print("steady:", steady_state)


## 3. Convert latency to useful throughput carefully

Throughput needs a meaningful numerator: examples, tokens, bytes, or approximate FLOPs.
FLOP/s can compare numerical kernels but may not predict application latency. Tokens/s
depends on model, batch, prompt/decode phase, context length, sampler, and runtime.


In [ ]:
m, k, n = a.shape[0], a.shape[1], b.shape[1]
operations = 2 * m * k * n
seconds = steady_state["median_ms"] / 1000
print("rough matmul throughput:", operations / seconds / 1e9, "GFLOP/s")

vector = torch.randn(2_000_000)
vector_stats = benchmark(lambda: vector + 1, repeats=30)
optimistic_bytes = vector.numel() * vector.element_size() * 2  # read + write
bandwidth = optimistic_bytes / (vector_stats["median_ms"] / 1000) / 1e9
print("vector add:", vector_stats, "optimistic effective GB/s:", bandwidth)


## 4. Allocation and reuse need equivalent semantics

Benchmark variants must compute the same result and expose equivalent synchronization.
Reused buffers change ownership and may be unsafe in a real graph; include correctness and
peak-live-memory checks, not only speed.


In [ ]:
left = np.ones(1_000_000, dtype=np.float32)
right = np.ones_like(left)
output = np.empty_like(left)
allocate = benchmark(lambda: left + right, repeats=50)
reuse = benchmark(lambda: np.add(left, right, out=output), repeats=50)
np.testing.assert_array_equal(output, left + right)
print("allocate:", allocate)
print("reuse:", reuse)


## 5. A reproducible report records the environment

At minimum record hardware, OS, Python/framework versions, device, dtype, shapes, command,
code revision, warm-up, repeats, synchronization, statistics, and correctness criteria.
Report distributions or percentiles rather than hiding variance behind one value.


In [ ]:
report = {
    "question": "steady-state CPU float32 matmul latency",
    "machine": platform.machine(),
    "platform": platform.platform(),
    "python": platform.python_version(),
    "torch": torch.__version__,
    "device": "cpu",
    "shape": [list(a.shape), list(b.shape)],
    "dtype": str(a.dtype),
    "warmup": 5,
    "metrics": steady_state,
    "correctness": "shape and finite output checked",
}
print(report)
assert report["metrics"]["repeats"] == 20


## Failure modes to recognize

- **Single-run result:** startup or OS noise determines the conclusion.
- **Async timing:** queued work completes outside the measured region.
- **Different semantics:** one variant omits transfers, allocation, or actual output work.
- **Metric substitution:** theoretical FLOPs are reported as user-perceived speed.

These are diagnostic patterns, not trivia. When a future model fails, reduce the problem to the smallest example in this notebook and test the relevant invariant.


## Deliberate practice

1. Extend the harness with p50, p95, mean, standard deviation, and raw-sample export.
2. Benchmark cold and steady matrix multiplication for three shapes and explain variance.
3. Write a Humor Machine inference benchmark specification covering load, prefill, first token, decode, and peak memory.

Do not merely rerun the provided cells. Make a prediction, change one variable, and write down why the result did or did not match your prediction.

## Exit ticket

**You are ready to continue when:** another engineer could reproduce your benchmark and understand exactly what was and was not included.

**Next:** 07 — Reading a model as a resource budget.
